In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Buscar todos los CSV montados por la competición
import pandas as pd, glob, re

csvs = glob.glob("/kaggle/input/titanic/**/*.csv", recursive=True)
print("CSV encontrados:")
for p in csvs: print(" -", p)

# Mapear posibles nombres (ES/EN)
def pick(patterns):
    pats = [re.compile(p, re.I) for p in patterns]
    for p in csvs:
        name = p.split("/")[-1]
        if any(r.search(name) for r in pats):
            return p
    return None

TRAIN_PATH  = pick([r"^train\.csv$", r"^tren\.csv$"])
TEST_PATH   = pick([r"^test\.csv$", r"^prueba\.csv$"])
SUBM_SAMPLE = pick([r"gender_submission\.csv", r"env[ií]o.*g[eé]nero\.csv"])

print("\nUsando rutas:")
print("train ->", TRAIN_PATH)
print("test  ->", TEST_PATH)
print("muestra envío ->", SUBM_SAMPLE)

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SUBM_SAMPLE)

train.shape, test.shape, train.head()


In [ ]:
import pandas as pd, glob, re

# Buscar todos los CSV montados por la competición


csvs = glob.glob("/kaggle/input/titanic/**/*.csv", recursive=True)
print("CSV encontrados:")
for p in csvs: print(" -", p)

# Mapear posibles nombres (ES/EN)
def pick(patterns):
    pats = [re.compile(p, re.I) for p in patterns]
    for p in csvs:
        name = p.split("/")[-1]
        if any(r.search(name) for r in pats):
            return p
    return None

TRAIN_PATH  = pick([r"^train\.csv$", r"^tren\.csv$"])
TEST_PATH   = pick([r"^test\.csv$", r"^prueba\.csv$"])
SUBM_SAMPLE = pick([r"gender_submission\.csv", r"env[ií]o.*g[eé]nero\.csv"])

print("\nUsando rutas:")
print("train ->", TRAIN_PATH)
print("test  ->", TEST_PATH)
print("muestra envío ->", SUBM_SAMPLE)

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SUBM_SAMPLE)

train.shape, test.shape, train.head()


In [ ]:
import pandas as pd, glob, re

# Buscar todos los CSV montados por la competición


csvs = glob.glob("/kaggle/input/titanic/**/*.csv", recursive=True)
print("CSV encontrados:")
for p in csvs: print(" -", p)

# Mapear posibles nombres (ES/EN)
def pick(patterns):
    pats = [re.compile(p, re.I) for p in patterns]
    for p in csvs:
        name = p.split("/")[-1]
        if any(r.search(name) for r in pats):
            return p
    return None

TRAIN_PATH  = pick([r"^train\.csv$", r"^tren\.csv$"])
TEST_PATH   = pick([r"^test\.csv$", r"^prueba\.csv$"])
SUBM_SAMPLE = pick([r"gender_submission\.csv", r"env[ií]o.*g[eé]nero\.csv"])

print("\nUsando rutas:")
print("train ->", TRAIN_PATH)
print("test  ->", TEST_PATH)
print("muestra envío ->", SUBM_SAMPLE)

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SUBM_SAMPLE)

train.shape, test.shape, train.head()


In [ ]:
train.dtypes

In [ ]:
train.describe()

In [ ]:
train["Survived"].value_counts(normalize=True)

In [ ]:
import os
print("cwd:", os.getcwd())
print("train.csv existe?", os.path.exists("train.csv"))
print("test.csv existe?", os.path.exists("test.csv"))


In [ ]:
# Antes (fallaba)
# train = pd.read_csv("train.csv")
# test  = pd.read_csv("test.csv")

# Ahora (Kaggle)
DATA_DIR = "/kaggle/input/titanic"
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")


In [ ]:
# Ruta fija de Titanic en Kaggle
DATA_DIR = "/kaggle/input/titanic"

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")

# Si quieres verificar:
print(train.shape, test.shape)
print(train.head())


In [ ]:
# =========================
# Titanic - TensorFlow (K-Fold + F1 threshold)
# Listo para Kaggle
# =========================
import os, re, glob, random
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow as tf

# ---------------------------
# Reproducibilidad
# ---------------------------
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# ---------------------------
# Cargar datos (Kaggle)
# ---------------------------
# Opción fija:
DATA_DIR = "/kaggle/input/titanic"

# (Opcional) autodetección por si cambia el nombre
if not os.path.exists(os.path.join(DATA_DIR, "train.csv")):
    candidates = glob.glob("/kaggle/input/**/train.csv", recursive=True)
    assert candidates, "Agrega el dataset: Add data → Competitions → Titanic."
    DATA_DIR = os.path.dirname(candidates[0])

print("Usando DATA_DIR:", DATA_DIR)
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
test_passenger_id = test["PassengerId"]

# ---------------------------
# Feature engineering
# ---------------------------
TITLE_MAP = {
    'mr':'Mr','mrs':'Mrs','miss':'Miss','master':'Master',
    'don':'Rare','rev':'Rare','dr':'Rare','mme':'Mrs','ms':'Miss',
    'major':'Rare','lady':'Rare','sir':'Rare','mlle':'Miss',
    'col':'Rare','capt':'Rare','countess':'Rare','jonkheer':'Rare'
}

def extract_title(name: str) -> str:
    if not isinstance(name, str):
        return "Unknown"
    m = re.search(r",\s*([^\.]+)\.", name)
    title = m.group(1).strip().lower() if m else "unknown"
    return TITLE_MAP.get(title, title.title())

def preprocess(df: pd.DataFrame, embarked_cats=None, title_cats=None) -> tuple[pd.DataFrame, dict]:
    out = df.copy()

    # Title
    out["Title"] = out["Name"].apply(extract_title)

    # Sex
    out["Sex"] = out["Sex"].map({"female": 1, "male": 0}).astype("float32")

    # Embarked categorías fijas
    if embarked_cats is None:
        embarked_cats = ["S","C","Q"]
    out["Embarked"] = out["Embarked"].fillna(out["Embarked"].mode()[0])\
                        .astype(pd.CategoricalDtype(categories=embarked_cats))
    embarked_dum = pd.get_dummies(out["Embarked"], prefix="Embarked")

    # Pclass one-hot
    pclass_dum = pd.get_dummies(out["Pclass"].astype("category"), prefix="Pclass")

    # Fare / Age
    out["Fare"] = out["Fare"].fillna(out["Fare"].median())
    out["Fare_log1p"] = np.log1p(out["Fare"])

    age_group = out.groupby(["Sex","Pclass"])["Age"].transform(lambda s: s.fillna(s.median()))
    out["Age"] = out["Age"].fillna(age_group).fillna(out["Age"].median())

    # Family
    out["FamilySize"] = out["SibSp"] + out["Parch"] + 1
    out["IsAlone"] = (out["FamilySize"] == 1).astype(int)
    out["HasRelatives"] = ((out["SibSp"] + out["Parch"]) > 0).astype(int)
    # Interacción simple
    out["Sex_Pclass3"] = (out["Sex"] * (out["Pclass"]==3).astype(int))

    # Title one-hot fijo
    if title_cats is None:
        title_cats = sorted(out["Title"].unique().tolist())
    out["Title"] = out["Title"].astype(pd.CategoricalDtype(categories=title_cats))
    title_dum = pd.get_dummies(out["Title"], prefix="Title")

    # Unir y limpiar
    out = pd.concat([out, embarked_dum, pclass_dum, title_dum], axis=1)
    drop_cols = ["PassengerId","Name","Ticket","Cabin","Embarked","Pclass","Title","Fare"]
    out = out.drop(columns=[c for c in drop_cols if c in out.columns])

    return out, {"embarked_cats": embarked_cats, "title_cats": title_cats}

# Ajustar categorías con TRAIN
X_full_tmp, cats = preprocess(train.drop(columns=["Survived"]))
X_test_tmp, _ = preprocess(test, embarked_cats=cats["embarked_cats"], title_cats=cats["title_cats"])

# Alinear columnas train/test
def align_cols(A: pd.DataFrame, B: pd.DataFrame):
    missing_in_B = set(A.columns) - set(B.columns)
    for c in missing_in_B:
        B[c] = 0
    extra_in_B = set(B.columns) - set(A.columns)
    if extra_in_B:
        B = B.drop(columns=list(extra_in_B))
    return B[A.columns]

X_full = X_full_tmp.copy()
X_test = align_cols(X_full_tmp, X_test_tmp)
y_full = train["Survived"].astype("int32").values

# ---------------------------
# Modelo (Keras)
# ---------------------------
def build_model(input_dim: int) -> keras.Model:
    norm = layers.Normalization()
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        norm,
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.25),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.15),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy", keras.metrics.AUC(name="auc")]
    )
    return model

# ---------------------------
# Umbral por F1 (helper)
# ---------------------------
def best_threshold_by_f1(y_true, y_prob, grid=None):
    y_true = np.asarray(y_true).astype(int).ravel()
    y_prob = np.asarray(y_prob).astype(float).ravel()
    if grid is None:
        grid = np.linspace(0.3, 0.7, 41)
    best_t, best_f1 = 0.5, -1.0
    for t in grid:
        y_hat = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, y_hat, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return float(best_t), float(best_f1)

# ---------------------------
# Entrenamiento K-Fold
# ---------------------------
K = 5
skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=SEED)

oof_pred = np.zeros(len(X_full))
test_pred = np.zeros(len(X_test))
thresholds = []

X_full_np = X_full.values.astype("float32")
X_test_np = X_test.values.astype("float32")

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_full_np, y_full), 1):
    X_tr, X_va = X_full_np[tr_idx], X_full_np[va_idx]
    y_tr, y_va = y_full[tr_idx], y_full[va_idx]

    model = build_model(X_tr.shape[1])

    # Adapt Normalization con train del fold
    for layer in model.layers:
        if isinstance(layer, layers.Normalization):
            layer.adapt(X_tr)

    early = keras.callbacks.EarlyStopping(patience=30, restore_best_weights=True,
                                          monitor="val_auc", mode="max")
    plateau = keras.callbacks.ReduceLROnPlateau(monitor="val_auc", mode="max",
                                                factor=0.5, patience=10, min_lr=1e-5, verbose=0)

    # Peso leve a clase positiva (ajusta si quieres)
    class_weight = {0: 1.0, 1: 1.2}

    model.fit(
        X_tr, y_tr,
        validation_data=(X_va, y_va),
        epochs=300,
        batch_size=64,
        verbose=1,
        callbacks=[early, plateau],
        class_weight=class_weight
    )

    p_va = model.predict(X_va).ravel()
    p_te = model.predict(X_test_np).ravel()

    best_thr, best_f1 = best_threshold_by_f1(y_va, p_va)
    thresholds.append(best_thr)
    oof_pred[va_idx] = p_va
    test_pred += p_te / K

    print(f"[Fold {fold}] AUC={roc_auc_score(y_va, p_va):.4f}  F1@best={best_f1:.4f}  thr={best_thr:.3f}")

# Métricas OOF
oof_auc = roc_auc_score(y_full, oof_pred)
oof_thr = float(np.median(thresholds))
oof_acc = accuracy_score(y_full, (oof_pred >= oof_thr).astype(int))
oof_f1  = f1_score(y_full, (oof_pred >= oof_thr).astype(int), zero_division=0)
print(f"OOF AUC={oof_auc:.4f}  OOF ACC@{oof_thr:.3f}={oof_acc:.4f}  OOF F1={oof_f1:.4f}")

# ---------------------------
# Submission (en /kaggle/working)
# ---------------------------
pred_test_label = (test_pred >= oof_thr).astype(int)
submission = pd.DataFrame({
    "PassengerId": test_passenger_id,
    "Survived": pred_test_label
})
submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Archivo '{submission_path}' generado en /kaggle/working ✅")


In [ ]:
import os
# Fuerza CPU (ignora GPU incluso si existe)
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"   # menos ruido de logs opcional
# (si aún da guerra con XLA)
os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=0"

import tensorflow as tf
# Seguridad extra: oculta dispositivos GPU a TF
try:
    tf.config.set_visible_devices([], "GPU")
except Exception:
    pass

# Verificación rápida
print("Dispositivos físicos:", tf.config.list_physical_devices())


In [ ]:
# =========================
# Titanic - TensorFlow (K-Fold + F1 threshold)
# Listo para Kaggle
# =========================
import os, re, glob, random
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow as tf

# ---------------------------
# Reproducibilidad
# ---------------------------
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# ---------------------------
# Cargar datos (Kaggle)
# ---------------------------
# Opción fija:
DATA_DIR = "/kaggle/input/titanic"

# (Opcional) autodetección por si cambia el nombre
if not os.path.exists(os.path.join(DATA_DIR, "train.csv")):
    candidates = glob.glob("/kaggle/input/**/train.csv", recursive=True)
    assert candidates, "Agrega el dataset: Add data → Competitions → Titanic."
    DATA_DIR = os.path.dirname(candidates[0])

print("Usando DATA_DIR:", DATA_DIR)
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
test_passenger_id = test["PassengerId"]

# ---------------------------
# Feature engineering
# ---------------------------
TITLE_MAP = {
    'mr':'Mr','mrs':'Mrs','miss':'Miss','master':'Master',
    'don':'Rare','rev':'Rare','dr':'Rare','mme':'Mrs','ms':'Miss',
    'major':'Rare','lady':'Rare','sir':'Rare','mlle':'Miss',
    'col':'Rare','capt':'Rare','countess':'Rare','jonkheer':'Rare'
}

def extract_title(name: str) -> str:
    if not isinstance(name, str):
        return "Unknown"
    m = re.search(r",\s*([^\.]+)\.", name)
    title = m.group(1).strip().lower() if m else "unknown"
    return TITLE_MAP.get(title, title.title())

def preprocess(df: pd.DataFrame, embarked_cats=None, title_cats=None) -> tuple[pd.DataFrame, dict]:
    out = df.copy()

    # Title
    out["Title"] = out["Name"].apply(extract_title)

    # Sex
    out["Sex"] = out["Sex"].map({"female": 1, "male": 0}).astype("float32")

    # Embarked categorías fijas
    if embarked_cats is None:
        embarked_cats = ["S","C","Q"]
    out["Embarked"] = out["Embarked"].fillna(out["Embarked"].mode()[0])\
                        .astype(pd.CategoricalDtype(categories=embarked_cats))
    embarked_dum = pd.get_dummies(out["Embarked"], prefix="Embarked")

    # Pclass one-hot
    pclass_dum = pd.get_dummies(out["Pclass"].astype("category"), prefix="Pclass")

    # Fare / Age
    out["Fare"] = out["Fare"].fillna(out["Fare"].median())
    out["Fare_log1p"] = np.log1p(out["Fare"])

    age_group = out.groupby(["Sex","Pclass"])["Age"].transform(lambda s: s.fillna(s.median()))
    out["Age"] = out["Age"].fillna(age_group).fillna(out["Age"].median())

    # Family
    out["FamilySize"] = out["SibSp"] + out["Parch"] + 1
    out["IsAlone"] = (out["FamilySize"] == 1).astype(int)
    out["HasRelatives"] = ((out["SibSp"] + out["Parch"]) > 0).astype(int)
    # Interacción simple
    out["Sex_Pclass3"] = (out["Sex"] * (out["Pclass"]==3).astype(int))

    # Title one-hot fijo
    if title_cats is None:
        title_cats = sorted(out["Title"].unique().tolist())
    out["Title"] = out["Title"].astype(pd.CategoricalDtype(categories=title_cats))
    title_dum = pd.get_dummies(out["Title"], prefix="Title")

    # Unir y limpiar
    out = pd.concat([out, embarked_dum, pclass_dum, title_dum], axis=1)
    drop_cols = ["PassengerId","Name","Ticket","Cabin","Embarked","Pclass","Title","Fare"]
    out = out.drop(columns=[c for c in drop_cols if c in out.columns])

    return out, {"embarked_cats": embarked_cats, "title_cats": title_cats}

# Ajustar categorías con TRAIN
X_full_tmp, cats = preprocess(train.drop(columns=["Survived"]))
X_test_tmp, _ = preprocess(test, embarked_cats=cats["embarked_cats"], title_cats=cats["title_cats"])

# Alinear columnas train/test
def align_cols(A: pd.DataFrame, B: pd.DataFrame):
    missing_in_B = set(A.columns) - set(B.columns)
    for c in missing_in_B:
        B[c] = 0
    extra_in_B = set(B.columns) - set(A.columns)
    if extra_in_B:
        B = B.drop(columns=list(extra_in_B))
    return B[A.columns]

X_full = X_full_tmp.copy()
X_test = align_cols(X_full_tmp, X_test_tmp)
y_full = train["Survived"].astype("int32").values

# ---------------------------
# Modelo (Keras)
# ---------------------------
def build_model(input_dim: int) -> keras.Model:
    norm = layers.Normalization()
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        norm,
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.25),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.15),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy", keras.metrics.AUC(name="auc")]
    )
    return model

# ---------------------------
# Umbral por F1 (helper)
# ---------------------------
def best_threshold_by_f1(y_true, y_prob, grid=None):
    y_true = np.asarray(y_true).astype(int).ravel()
    y_prob = np.asarray(y_prob).astype(float).ravel()
    if grid is None:
        grid = np.linspace(0.3, 0.7, 41)
    best_t, best_f1 = 0.5, -1.0
    for t in grid:
        y_hat = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, y_hat, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return float(best_t), float(best_f1)

# ---------------------------
# Entrenamiento K-Fold
# ---------------------------
K = 5
skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=SEED)

oof_pred = np.zeros(len(X_full))
test_pred = np.zeros(len(X_test))
thresholds = []

X_full_np = X_full.values.astype("float32")
X_test_np = X_test.values.astype("float32")

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_full_np, y_full), 1):
    X_tr, X_va = X_full_np[tr_idx], X_full_np[va_idx]
    y_tr, y_va = y_full[tr_idx], y_full[va_idx]

    model = build_model(X_tr.shape[1])

    # Adapt Normalization con train del fold
    for layer in model.layers:
        if isinstance(layer, layers.Normalization):
            layer.adapt(X_tr)

    early = keras.callbacks.EarlyStopping(patience=30, restore_best_weights=True,
                                          monitor="val_auc", mode="max")
    plateau = keras.callbacks.ReduceLROnPlateau(monitor="val_auc", mode="max",
                                                factor=0.5, patience=10, min_lr=1e-5, verbose=0)

    # Peso leve a clase positiva (ajusta si quieres)
    class_weight = {0: 1.0, 1: 1.2}

    model.fit(
        X_tr, y_tr,
        validation_data=(X_va, y_va),
        epochs=300,
        batch_size=64,
        verbose=1,
        callbacks=[early, plateau],
        class_weight=class_weight
    )

    p_va = model.predict(X_va).ravel()
    p_te = model.predict(X_test_np).ravel()

    best_thr, best_f1 = best_threshold_by_f1(y_va, p_va)
    thresholds.append(best_thr)
    oof_pred[va_idx] = p_va
    test_pred += p_te / K

    print(f"[Fold {fold}] AUC={roc_auc_score(y_va, p_va):.4f}  F1@best={best_f1:.4f}  thr={best_thr:.3f}")

# Métricas OOF
oof_auc = roc_auc_score(y_full, oof_pred)
oof_thr = float(np.median(thresholds))
oof_acc = accuracy_score(y_full, (oof_pred >= oof_thr).astype(int))
oof_f1  = f1_score(y_full, (oof_pred >= oof_thr).astype(int), zero_division=0)
print(f"OOF AUC={oof_auc:.4f}  OOF ACC@{oof_thr:.3f}={oof_acc:.4f}  OOF F1={oof_f1:.4f}")

# ---------------------------
# Submission (en /kaggle/working)
# ---------------------------
pred_test_label = (test_pred >= oof_thr).astype(int)
submission = pd.DataFrame({
    "PassengerId": test_passenger_id,
    "Survived": pred_test_label
})
submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Archivo '{submission_path}' generado en /kaggle/working ✅")


In [ ]:
# ================================
# Listar NOMBRES y APELLIDOS de los PassengerId
# que tu modelo predijo como Survived=1 (test set)
# ================================
import os, re, glob
import pandas as pd
import numpy as np

# --- 1) Localizar datos de Titanic en Kaggle ---
DATA_DIR = "/kaggle/input/titanic"
if not os.path.exists(os.path.join(DATA_DIR, "test.csv")):
    candidates = glob.glob("/kaggle/input/**/test.csv", recursive=True)
    assert candidates, "No se encontró test.csv en /kaggle/input. Agrega el dataset (Add data → Competitions → Titanic)."
    DATA_DIR = os.path.dirname(candidates[0])

print("Usando DATA_DIR:", DATA_DIR)

# --- 2) Cargar test (tiene nombres) y submission (tiene tus predicciones) ---
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))           # contiene PassengerId y Name
sub  = pd.read_csv("/kaggle/working/submission.csv")             # contiene PassengerId y Survived predicho

# Seguridad básica
assert {"PassengerId","Survived"}.issubset(sub.columns), "submission.csv debe tener columnas PassengerId y Survived"
assert {"PassengerId","Name"}.issubset(test.columns), "test.csv debe tener columnas PassengerId y Name"

# --- 3) Unir por PassengerId y filtrar Survived==1 ---
merged = test.merge(sub, on="PassengerId", how="inner")
survivors = merged[merged["Survived"] == 1].copy()

# --- 4) Funciones para extraer Apellido y Nombres del formato Kaggle ---
# Formato típico: "Braund, Mr. Owen Harris"
# Casos con paréntesis (mujeres casadas): "Cumings, Mrs. John Bradley (Florence Briggs Thayer)"
#   - Apellido = antes de la coma
#   - Nombres: si hay paréntesis, usar el contenido dentro de paréntesis; si no, lo que va tras el título

title_pattern = re.compile(r",\s*([^\.]+)\.")  # captura el título (Mr, Mrs, Miss, Master, etc.)

def split_name(raw: str):
    if not isinstance(raw, str):
        return {"LastName": np.nan, "FirstNames": np.nan, "Title": np.nan}
    # Apellido: antes de la coma
    last = raw.split(",")[0].strip()
    # Título:
    m = title_pattern.search(raw)
    title = m.group(1).strip() if m else ""
    # Si hay paréntesis, tomar lo de adentro como nombres
    if "(" in raw and ")" in raw:
        inside = raw.split("(", 1)[1].rsplit(")", 1)[0].strip()
        firsts = inside
    else:
        # Lo que viene tras "Title." son los nombres (pueden tener segundos nombres)
        after = raw.split(",", 1)[1] if "," in raw else ""
        after = after.split(".", 1)[1].strip() if "." in after else after.strip()
        firsts = after
    return {"LastName": last, "FirstNames": firsts, "Title": title}

parsed = survivors["Name"].apply(split_name).apply(pd.Series)
survivors = pd.concat([survivors, parsed], axis=1)

# --- 5) Seleccionar columnas bonitas y ordenar ---
out_cols = ["PassengerId", "Survived", "LastName", "FirstNames", "Title", "Name"]
survivors_out = survivors[out_cols].sort_values(["LastName","FirstNames"]).reset_index(drop=True)

# --- 6) Guardar resultado para descargar / revisar ---
out_path = "/kaggle/working/survivors_with_names.csv"
survivors_out.to_csv(out_path, index=False)
print(f"Archivo generado: {out_path}")
print(f"Total predichos como sobrevivientes: {len(survivors_out)}")
print(survivors_out.head(10))



In [ ]:
import os
print(os.listdir("/kaggle/working"))


In [ ]:
import pandas as pd

submission = pd.read_csv("/kaggle/working/submission.csv")
print(submission.shape)
print(submission.columns)
submission.head()
